In [1]:
import json

notebook_json = {
  "cells": [
    {
      "cell_type": "markdown",
      "id": "2c99a4cb",
      "metadata": {
        "id": "2c99a4cb"
      },
      "source": [
        "# S4 · AndinaLog 03B · Notebook 1 · Diagnóstico de telemetría IoT\n",
        "\n",
        "Este notebook trabaja **solo** con `andinalog_iot_telemetry.csv`. Lee la capa Bronze desde `datasets/AndinaLog_03B_Bronce/`, detecta problemas y conserva los diez campos originales. No convierte unidades, no imputa ni corrige datos. El tratamiento de los casos recuperables corresponde al notebook 2, después de aprobar sus reglas.\n",
        "\n",
        "Cada ejecución reemplaza cuatro archivos en `S4/andinalog_iot_telemetry/notebook1/salidas/`:\n",
        "\n",
        "1. `andinalog_iot_telemetry_diagnosticado.csv`: todas las filas, los campos originales y solo `fila_bronze`, `en_cuarentena` y `columnas_con_problemas`.\n",
        "2. `andinalog_iot_telemetry_problemas.csv`: una fila por problema, con columna, código estable y evidencia.\n",
        "3. `andinalog_iot_telemetry_cuarentena.csv`: extracto informativo de las filas marcadas.\n",
        "4. `andinalog_iot_telemetry_reporte_calidad.csv`: conteos y huella SHA-256 del CSV de origen.\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "4efb6404",
      "metadata": {
        "id": "4efb6404"
      },
      "source": [
        "## 1 · Configuración y origen\n",
        "\n",
        "En local, ejecuta el notebook desde cualquier carpeta dentro del proyecto. En Colab, monta Drive, selecciona `ENTORNO = \"drive\"` y ajusta `RUTA_PROYECTO_DRIVE` a la carpeta que contiene `datasets/` y `S4/`. Las salidas van a `S4/andinalog_iot_telemetry/notebook1/salidas/` en ese mismo entorno.\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "id": "a96c0f93",
      "metadata": {
        "id": "a96c0f93"
      },
      "outputs": [],
      "source": [
        "from pathlib import Path\n",
        "import hashlib\n",
        "import os\n",
        "import re\n",
        "import tempfile\n",
        "import pandas as pd\n",
        "\n",
        "ENTORNO = \"drive\"  # \"local\" o \"drive\"\n",
        "RUTA_PROYECTO_DRIVE = \"/content/drive/MyDrive/GIAD\"  # ajustar si la carpeta real es otra\n",
        "CARPETA_DATASETS = \"AndinaLog_03B_Bronce\"\n",
        "NOMBRE_CSV = \"andinalog_iot_telemetry.csv\"\n",
        "VERSION_DIAGNOSTICO = \"GIAD-M3-S4-IOT-diagnostico-v2\"\n",
        "\n",
        "COLUMNAS_ORIGINALES = [\n",
        "    \"timestamp\", \"viaje_id\", \"order_id\", \"camion_id\", \"producto_id\",\n",
        "    \"temperatura_cabina_c\", \"temp_unit\", \"humedad_cabina_pct\",\n",
        "    \"desviacion_termica_flag\", \"desviacion_proximos_60min_flag\",\n",
        "]\n",
        "\n",
        "def encontrar_raiz_local():\n",
        "    for carpeta in [Path.cwd(), *Path.cwd().parents]:\n",
        "        if (carpeta / \"datasets\" / CARPETA_DATASETS).is_dir() and (carpeta / \"S4\").is_dir():\n",
        "            return carpeta\n",
        "    raise FileNotFoundError(\"No se encontró la raíz del proyecto; ejecuta dentro de practicasNotebookColab.\")\n",
        "\n",
        "def configurar_rutas(entorno, ruta_drive):\n",
        "    if entorno == \"drive\":\n",
        "        from google.colab import drive\n",
        "        drive.mount(\"/content/drive\")\n",
        "        raiz = Path(ruta_drive)\n",
        "    elif entorno == \"local\":\n",
        "        raiz = encontrar_raiz_local()\n",
        "    else:\n",
        "        raise ValueError(\"ENTORNO debe ser 'local' o 'drive'\")\n",
        "    bronze = raiz / \"datasets\" / CARPETA_DATASETS / NOMBRE_CSV\n",
        "    salidas = raiz / \"S4\" / \"andinalog_iot_telemetry\"/ \"notebook1\" / \"salidas\"\n",
        "    if not bronze.is_file():\n",
        "        raise FileNotFoundError(f\"No se encontró el CSV Bronze: {bronze}\")\n",
        "    return bronze, salidas\n",
        "\n",
        "RUTA_BRONZE, DIRECTORIO_SALIDAS = configurar_rutas(ENTORNO, RUTA_PROYECTO_DRIVE)\n",
        "print(\"Bronze:\", RUTA_BRONZE)\n",
        "print(\"Salidas:\", DIRECTORIO_SALIDAS)\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "84cbe589",
      "metadata": {
        "id": "84cbe589"
      },
      "source": [
        "## 2 · Carga y contrato\n",
        "\n",
        "La lectura mantiene todas las columnas como texto y los vacíos como cadenas vacías. Las conversiones numéricas y de fecha usadas para comprobar errores son temporales: no se exportan como valores transformados.\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "id": "f42a3ef2",
      "metadata": {
        "id": "f42a3ef2"
      },
      "outputs": [],
      "source": [
        "def cargar_bronze(ruta):\n",
        "    huella = hashlib.sha256(ruta.read_bytes()).hexdigest()\n",
        "    df = pd.read_csv(ruta, dtype=\"string\", encoding=\"utf-8-sig\", keep_default_na=False)\n",
        "    return df, huella\n",
        "\n",
        "def validar_esquema(df):\n",
        "    if list(df.columns) != COLUMNAS_ORIGINALES:\n",
        "        faltantes = sorted(set(COLUMNAS_ORIGINALES) - set(df.columns))\n",
        "        extras = sorted(set(df.columns) - set(COLUMNAS_ORIGINALES))\n",
        "        raise ValueError(f\"Esquema inesperado. Faltantes: {faltantes}; extras: {extras}; orden: {list(df.columns)}\")\n",
        "    if not df.columns.is_unique:\n",
        "        raise ValueError(\"Hay nombres de columnas duplicados\")\n",
        "    return df\n",
        "\n",
        "df_bronze, HASH_BRONZE = cargar_bronze(RUTA_BRONZE)\n",
        "validar_esquema(df_bronze)\n",
        "print(f\"Bronze: {len(df_bronze):,} filas × {len(df_bronze.columns)} columnas\")\n",
        "print(\"SHA-256:\", HASH_BRONZE)\n",
        "display(df_bronze.head())\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "af010445",
      "metadata": {
        "id": "af010445"
      },
      "source": [
        "## 3 · Catálogo y reglas de diagnóstico\n",
        "\n",
        "Los códigos son estables para que el notebook 2 pueda reconocer el problema exacto. `columna_afectada` puede nombrar una sola columna o una clave compuesta, como `viaje_id+timestamp`. Las reglas detectan; no deciden todavía cómo corregir.\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "id": "5146defc",
      "metadata": {
        "id": "5146defc"
      },
      "outputs": [],
      "source": [
        "CATALOGO_PROBLEMAS = pd.DataFrame([\n",
        "    (\"timestamp\", \"FECHA_INVALIDA\", \"No cumple el formato o no existe en el calendario\"),\n",
        "    (\"viaje_id\", \"FALTANTE\", \"Identificador vacío\"),\n",
        "    (\"order_id\", \"FALTANTE\", \"Identificador vacío\"),\n",
        "    (\"camion_id\", \"FALTANTE\", \"Identificador vacío\"),\n",
        "    (\"producto_id\", \"FALTANTE\", \"Identificador vacío\"),\n",
        "    (\"viaje_id+timestamp\", \"DUPLICADO\", \"Clave de lectura repetida; se marca la aparición posterior\"),\n",
        "    (\"temp_unit\", \"UNIDAD_NO_RECONOCIDA\", \"Unidad distinta de C o F\"),\n",
        "    (\"temperatura_cabina_c\", \"FALTANTE\", \"Campo vacío\"),\n",
        "    (\"temperatura_cabina_c\", \"NO_NUMERICA\", \"Valor no convertible a número\"),\n",
        "    (\"humedad_cabina_pct\", \"FALTANTE\", \"Campo vacío\"),\n",
        "    (\"humedad_cabina_pct\", \"NO_NUMERICA\", \"Valor no convertible a número\"),\n",
        "    (\"humedad_cabina_pct\", \"FUERA_RANGO\", \"Humedad relativa fuera de 0 a 100%\"),\n",
        "    (\"desviacion_termica_flag\", \"FLAG_INVALIDA\", \"Valor distinto de 0 o 1\"),\n",
        "    (\"desviacion_proximos_60min_flag\", \"FLAG_INVALIDA\", \"Valor distinto de 0 o 1\"),\n",
        "], columns=[\"columna_afectada\", \"codigo_error\", \"criterio\"])\n",
        "display(CATALOGO_PROBLEMAS)\n",
        "\n",
        "def registrar_problema(df, mascara, columna, codigo, evidencia=None):\n",
        "    mascara = mascara.fillna(False).astype(bool)\n",
        "    filas = df.loc[mascara, [\"fila_bronze\"]].copy()\n",
        "    filas[\"columna_afectada\"] = columna\n",
        "    filas[\"codigo_error\"] = codigo\n",
        "    if evidencia is None:\n",
        "        evidencia = df[columna] if columna in df.columns else pd.Series(\"\", index=df.index, dtype=\"string\")\n",
        "    filas[\"valor_original\"] = evidencia.loc[mascara].astype(\"string\").to_numpy()\n",
        "    return filas\n",
        "\n",
        "def texto(df, columna):\n",
        "    return df[columna].astype(\"string\").str.strip()\n",
        "\n",
        "def detectar_identificadores(df):\n",
        "    return [registrar_problema(df, texto(df, col).eq(\"\"), col, \"FALTANTE\")\n",
        "            for col in [\"viaje_id\", \"order_id\", \"camion_id\", \"producto_id\"]]\n",
        "\n",
        "def detectar_timestamps_y_duplicados(df):\n",
        "    ts = texto(df, \"timestamp\")\n",
        "    formato = ts.str.fullmatch(r\"\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}\").fillna(False)\n",
        "    fecha = pd.to_datetime(ts, format=\"%Y-%m-%d %H:%M:%S\", errors=\"coerce\")\n",
        "    invalida = ~formato | fecha.isna()\n",
        "    clave = pd.DataFrame({\"viaje_id\": texto(df, \"viaje_id\"), \"timestamp\": ts})\n",
        "    duplicada = clave.duplicated(keep=\"first\")\n",
        "    evidencia_clave = texto(df, \"viaje_id\") + \" + \" + ts\n",
        "    return [\n",
        "        registrar_problema(df, invalida, \"timestamp\", \"FECHA_INVALIDA\"),\n",
        "        registrar_problema(df, duplicada, \"viaje_id+timestamp\", \"DUPLICADO\", evidencia_clave),\n",
        "    ]\n",
        "\n",
        "def detectar_medicion(df, columna, rango=None):\n",
        "    valor = texto(df, columna)\n",
        "    numero = pd.to_numeric(valor, errors=\"coerce\")\n",
        "    hallazgos = [\n",
        "        registrar_problema(df, valor.eq(\"\"), columna, \"FALTANTE\"),\n",
        "        registrar_problema(df, valor.ne(\"\") & numero.isna(), columna, \"NO_NUMERICA\"),\n",
        "    ]\n",
        "    if rango is not None:\n",
        "        minimo, maximo = rango\n",
        "        hallazgos.append(registrar_problema(df, numero.notna() & ~numero.between(minimo, maximo), columna, \"FUERA_RANGO\"))\n",
        "    return hallazgos\n",
        "\n",
        "def detectar_unidad_y_flags(df):\n",
        "    unidad = texto(df, \"temp_unit\").str.upper()\n",
        "    hallazgos = [registrar_problema(df, ~unidad.isin([\"C\", \"F\"]), \"temp_unit\", \"UNIDAD_NO_RECONOCIDA\")]\n",
        "    for col in [\"desviacion_termica_flag\", \"desviacion_proximos_60min_flag\"]:\n",
        "        valor = pd.to_numeric(texto(df, col), errors=\"coerce\")\n",
        "        hallazgos.append(registrar_problema(df, ~valor.isin([0, 1]), col, \"FLAG_INVALIDA\"))\n",
        "    return hallazgos\n",
        "\n",
        "def diagnosticar(df_bronze):\n",
        "    principal = df_bronze.copy(deep=True)\n",
        "    principal.insert(0, \"fila_bronze\", range(1, len(principal) + 1))\n",
        "    hallazgos = (\n",
        "        detectar_identificadores(principal)\n",
        "        + detectar_timestamps_y_duplicados(principal)\n",
        "        + detectar_medicion(principal, \"temperatura_cabina_c\")\n",
        "        + detectar_medicion(principal, \"humedad_cabina_pct\", rango=(0, 100))\n",
        "        + detectar_unidad_y_flags(principal)\n",
        "    )\n",
        "    problemas = pd.concat(hallazgos, ignore_index=True)\n",
        "    problemas = problemas.sort_values([\"fila_bronze\", \"columna_afectada\", \"codigo_error\"], kind=\"stable\").reset_index(drop=True)\n",
        "    problemas[\"version_diagnostico\"] = VERSION_DIAGNOSTICO\n",
        "    columnas_por_fila = problemas.groupby(\"fila_bronze\")[\"columna_afectada\"].agg(\n",
        "        lambda valores: \"|\".join(dict.fromkeys(valores))\n",
        "    )\n",
        "    principal[\"columnas_con_problemas\"] = principal[\"fila_bronze\"].map(columnas_por_fila).fillna(\"\")\n",
        "    principal[\"en_cuarentena\"] = principal[\"columnas_con_problemas\"].ne(\"\")\n",
        "    return principal, problemas\n",
        "\n",
        "df_diagnosticado, df_problemas = diagnosticar(df_bronze)\n",
        "df_cuarentena = df_diagnosticado.loc[df_diagnosticado[\"en_cuarentena\"]].copy()\n",
        "print(f\"Principal: {len(df_diagnosticado):,}; problemas: {len(df_problemas):,}; filas en cuarentena: {len(df_cuarentena):,}\")\n",
        "display(df_problemas.groupby([\"columna_afectada\", \"codigo_error\"]).size().rename(\"filas\").reset_index())\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "7d3e8320",
      "metadata": {
        "id": "7d3e8320"
      },
      "source": [
        "## 4 · Reporte y comprobaciones antes de exportar\n",
        "\n",
        "El reporte registra la huella SHA-256 para reconocer la versión exacta del CSV de origen. Los conteos de problemas pueden superar el número de filas en cuarentena porque una fila puede tener varios hallazgos.\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "id": "78d3cb9b",
      "metadata": {
        "id": "78d3cb9b"
      },
      "outputs": [],
      "source": [
        "def construir_reporte(df_bronze, principal, problemas, ruta, huella):\n",
        "    conteos = problemas.groupby([\"columna_afectada\", \"codigo_error\"]).size()\n",
        "    datos = [\n",
        "        (\"archivo_bronze\", ruta.name),\n",
        "        (\"sha256_bronze\", huella),\n",
        "        (\"version_diagnostico\", VERSION_DIAGNOSTICO),\n",
        "        (\"filas_bronze\", len(df_bronze)),\n",
        "        (\"filas_diagnosticadas\", len(principal)),\n",
        "        (\"filas_en_cuarentena\", int(principal[\"en_cuarentena\"].sum())),\n",
        "        (\"filas_sin_cuarentena\", int((~principal[\"en_cuarentena\"]).sum())),\n",
        "        (\"problemas_detectados\", len(problemas)),\n",
        "    ]\n",
        "    datos += [(f\"{col}:{codigo}\", int(total)) for (col, codigo), total in conteos.items()]\n",
        "    return pd.DataFrame(datos, columns=[\"metrica\", \"valor\"])\n",
        "\n",
        "def validar_resultados(df_bronze, principal, problemas, cuarentena, reporte):\n",
        "    assert list(principal.columns) == [\"fila_bronze\", *COLUMNAS_ORIGINALES, \"columnas_con_problemas\", \"en_cuarentena\"]\n",
        "    pd.testing.assert_frame_equal(principal[COLUMNAS_ORIGINALES], df_bronze[COLUMNAS_ORIGINALES])\n",
        "    assert len(principal) == len(df_bronze)\n",
        "    assert principal[\"fila_bronze\"].is_unique\n",
        "    assert len(cuarentena) == int(principal[\"en_cuarentena\"].sum())\n",
        "    assert problemas[\"fila_bronze\"].isin(principal[\"fila_bronze\"]).all()\n",
        "    assert problemas[[\"columna_afectada\", \"codigo_error\"]].apply(tuple, axis=1).isin(\n",
        "        CATALOGO_PROBLEMAS[[\"columna_afectada\", \"codigo_error\"]].apply(tuple, axis=1)\n",
        "    ).all()\n",
        "    assert set(problemas[\"fila_bronze\"]) == set(cuarentena[\"fila_bronze\"])\n",
        "    assert len(reporte) >= 8\n",
        "\n",
        "reporte_calidad = construir_reporte(df_bronze, df_diagnosticado, df_problemas, RUTA_BRONZE, HASH_BRONZE)\n",
        "validar_resultados(df_bronze, df_diagnosticado, df_problemas, df_cuarentena, reporte_calidad)\n",
        "display(reporte_calidad)\n",
        "print(\"Comprobaciones previas a la exportación: correctas\")\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "792cf163",
      "metadata": {
        "id": "792cf163"
      },
      "source": [
        "## 5 · Exportación reproducible\n",
        "\n",
        "Los cuatro CSV se escriben primero como archivos temporales en `S4/andinalog_iot_telemetry/notebook1/salidas/` y se reemplazan con el mismo nombre al final. El CSV Bronze nunca se sobrescribe. Si se vuelve a ejecutar con la misma fuente y reglas, las salidas se actualizan en lugar de acumular versiones antiguas.\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "id": "459ed35d",
      "metadata": {
        "id": "459ed35d"
      },
      "outputs": [],
      "source": [
        "def exportar_salidas(directorio, tablas, ruta_bronze, huella_inicial):\n",
        "    if hashlib.sha256(ruta_bronze.read_bytes()).hexdigest() != huella_inicial:\n",
        "        raise RuntimeError(\"El CSV Bronze cambió durante la ejecución; no se exportarán resultados\")\n",
        "    directorio.mkdir(parents=True, exist_ok=True)\n",
        "    temporales = {}\n",
        "    try:\n",
        "        for nombre, tabla in tablas.items():\n",
        "            destino = directorio / nombre\n",
        "            with tempfile.NamedTemporaryFile(mode=\"w\", suffix=\".csv\", prefix=\".tmp_iot_\", dir=directorio,\n",
        "                                             encoding=\"utf-8-sig\", newline=\"\", delete=False) as tmp:\n",
        "                tabla.to_csv(tmp, index=False)\n",
        "                temporales[destino] = Path(tmp.name)\n",
        "        for destino, temporal in temporales.items():\n",
        "            os.replace(temporal, destino)\n",
        "    finally:\n",
        "        for temporal in temporales.values():\n",
        "            temporal.unlink(missing_ok=True)\n",
        "    return list(temporales)\n",
        "\n",
        "tablas_salida = {\n",
        "    \"andinalog_iot_telemetry_diagnosticado.csv\": df_diagnosticado,\n",
        "    \"andinalog_iot_telemetry_problemas.csv\": df_problemas,\n",
        "    \"andinalog_iot_telemetry_cuarentena.csv\": df_cuarentena,\n",
        "    \"andinalog_iot_telemetry_reporte_calidad.csv\": reporte_calidad,\n",
        "}\n",
        "rutas_creadas = exportar_salidas(DIRECTORIO_SALIDAS, tablas_salida, RUTA_BRONZE, HASH_BRONZE)\n",
        "for ruta in rutas_creadas:\n",
        "    print(ruta)\n",
        "print(\"Bronze intacta; salidas anteriores reemplazadas\")\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "77f38a6e",
      "metadata": {
        "id": "77f38a6e"
      },
      "source": [
        "## Siguiente etapa\n",
        "\n",
        "El notebook 2 leerá el archivo diagnosticado y el detalle de problemas. El informe de S4 justifica tratamientos posibles, pero ninguna regla de curación está aprobada automáticamente por este diagnóstico. Una fila saldrá de cuarentena solo cuando todos sus problemas hayan sido resueltos y validados.\n"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "codemirror_mode": {
        "name": "ipython",
        "version": 3
      },
      "file_extension": ".py",
      "mimetype": "text/x-python",
      "name": "python",
      "nbconvert_exporter": "python",
      "pygments_lexer": "ipython3",
      "version": "3.10.0"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}

json_str = json.dumps(notebook_json, indent=2, ensure_ascii=False)
print("Length of formatted json:", len(json_str))

Length of formatted json: 19367
